<a href="https://colab.research.google.com/github/rantawadeesritakorn-tech/Project-Hotel/blob/%E0%B8%84%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88-4/%E0%B8%84%E0%B8%99%E0%B8%97%E0%B8%B5%E0%B9%88_4.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## ส่วนที่ 6 — บันทึกข้อมูลลง CSV และ SQLite

ออกแบบเป็น 3 ตารางที่สัมพันธ์กัน: `guests` และ `rooms` เชื่อมกันผ่าน `bookings`



In [ ]:
def export_dataframes(hotel):
    """แปลง object ทั้งหมดเป็น DataFrame 3 ตัว (guests / rooms / bookings)"""
    guests_df = pd.DataFrame([{
        "guest_id": g.guest_id,
        "name": g.name,
        "phone": g.phone,
        "nationality": g.nationality,
        "member_tier": g.member_tier,
        "stay_count": g.stay_count,
    } for g in hotel.guests.values()])

    rooms_df = pd.DataFrame([{
        "room_id": r.room_id,
        "room_number": r.room_number,
        "room_type": r.room_type,
        "base_price": r.base_price,
        "capacity": r.capacity,
        "floor": r.floor,
        "nights_sold": r.nights_sold(),
    } for r in hotel.rooms])

    # list comprehension เรียก method to_dict() ของ object แต่ละใบ
    bookings_df = pd.DataFrame([b.to_dict() for b in hotel.bookings])
    return guests_df, rooms_df, bookings_df


def save_csv(guests_df, rooms_df, bookings_df, folder="data"):
    """บันทึก CSV 3 ไฟล์ลงโฟลเดอร์ data/"""
    os.makedirs(folder, exist_ok=True)
    paths = {}
    for name, df in [("hotel_guests", guests_df), ("hotel_rooms", rooms_df),
                     ("hotel_bookings", bookings_df)]:
        path = os.path.join(folder, f"{name}.csv")
        df.to_csv(path, index=False, encoding="utf-8-sig")
        paths[name] = path
    return paths


def save_sqlite(guests_df, rooms_df, bookings_df, db_path="data/hotel.db"):
    """สร้างฐานข้อมูล SQLite 3 ตาราง แล้ว return connection"""
    os.makedirs(os.path.dirname(db_path), exist_ok=True)
    conn = sqlite3.connect(db_path)
    guests_df.to_sql("guests", conn, if_exists="replace", index=False)
    rooms_df.to_sql("rooms", conn, if_exists="replace", index=False)
    bookings_df.to_sql("bookings", conn, if_exists="replace", index=False)
    conn.commit()
    return conn

In [ ]:
guests_df, rooms_df, bookings_df = export_dataframes(hotel)
paths = save_csv(guests_df, rooms_df, bookings_df)
conn = save_sqlite(guests_df, rooms_df, bookings_df)

print("บันทึกไฟล์เรียบร้อย")
for p in paths.values():
    print("  ", p)
print("   data/hotel.db")
print("\nขนาดตาราง | bookings:", bookings_df.shape,
      "| guests:", guests_df.shape, "| rooms:", rooms_df.shape)
bookings_df.head()

บันทึกไฟล์เรียบร้อย
   data/hotel_guests.csv
   data/hotel_rooms.csv
   data/hotel_bookings.csv
   data/hotel.db

ขนาดตาราง | bookings: (960, 31) | guests: (748, 6) | rooms: (15, 7)


,booking_id,guest_id,room_id,room_number,room_type,booking_date,check_in,check_out,nights,lead_time_days,...,cancel_date,rate_per_night,room_charge,extra_charge,discount,service_charge,vat,total_price,commission,net_revenue
0,1,1,5,105,Standard,2024-09-07,2025-01-31,2025-02-01,1,146,...,,"1,610.00","1,610.00",0,0.00,161.00,123.97,"1,894.97",284.25,"1,610.72"
1,2,2,10,110,Deluxe,2024-09-08,2025-03-07,2025-03-08,1,180,...,2025-02-12,"2,024.00","2,024.00",0,0.00,202.40,155.85,"2,382.25",357.34,"2,024.91"
2,3,2,13,203,Suite,2024-09-16,2025-01-18,2025-01-26,8,124,...,,"4,830.00","38,640.00",8400,0.00,"4,704.00","3,622.08","55,366.08","8,304.91","47,061.17"
3,4,3,1,101,Standard,2024-09-20,2025-01-10,2025-01-16,6,112,...,,"1,610.00","9,660.00",2100,0.00,"1,176.00",905.52,"13,841.52",276.83,"13,564.69"
4,5,4,6,106,Standard,2024-09-20,2025-01-02,2025-01-14,12,104,...,,"1,610.00","19,320.00",4200,0.00,"2,352.00","1,811.04","27,683.04","4,152.46","23,530.58"


## ส่วนที่ 7 — วิเคราะห์ด้วย SQL

**Query ที่เขียนทั้งหมด 7 query**
- Query 1–3 : SELECT / WHERE / ORDER BY
- Query 4 : GROUP BY + aggregate function (COUNT / SUM / AVG)
- Query 5–6 : JOIN (Query 5 เป็นการ JOIN 3 ตารางพร้อมกัน)
- Query 7 : CASE WHEN + GROUP BY

รันผ่านฐานข้อมูล SQLite (`data/hotel.db`) ด้วย `pandas.read_sql_query()` แสดงผลทุก query เป็น DataFrame


In [ ]:
# ตัวรัน SQL หลักของงานนี้ : อ่านจากฐานข้อมูล SQLite (data/hotel.db) ที่สร้างไว้ในส่วนที่ 6
# ใช้ pandas.read_sql_query() แสดงผลลัพธ์ของทุก query เป็น DataFrame

def q(sql):
    """รัน SQL บนฐานข้อมูล SQLite แล้วคืนผลลัพธ์เป็น DataFrame

    {t} และ {e} เป็นตัวคั่นชื่อตาราง ทำให้ query ชุดเดียวกันนำไปรันบน
    BigQuery ได้ด้วยโดยไม่ต้องแก้ (ดูส่วนเสริมท้ายหัวข้อนี้)
    """
    full = sql.replace("{t}", "").replace("{e}", "")
    return pd.read_sql_query(full, conn)


# ตรวจว่าตารางในฐานข้อมูลครบ 3 ตารางก่อนเริ่ม query
tables = pd.read_sql_query(
    "SELECT name FROM sqlite_master WHERE type='table' ORDER BY name", conn
)
print("ตารางในฐานข้อมูล data/hotel.db :", ", ".join(tables["name"]))
for t in tables["name"]:
    n = pd.read_sql_query(f"SELECT COUNT(*) AS n FROM {t}", conn)["n"][0]
    print(f"   {t:10s} {n:,} แถว")


ตารางในฐานข้อมูล data/hotel.db : bookings, guests, rooms
   bookings   960 แถว
   guests     748 แถว
   rooms      15 แถว


**Query 1** — SELECT / WHERE / ORDER BY : ใบจองห้อง Suite ที่ยอดสูงสุด 10 อันดับแรก

In [ ]:
q1 = q("""
    SELECT booking_id, room_number, check_in, nights, channel,
           ROUND(total_price, 2) AS total_price
    FROM {t}bookings{e}
    WHERE room_type = 'Suite' AND status != 'CANCELLED'
    ORDER BY total_price DESC
    LIMIT 10
""")
q1

,booking_id,room_number,check_in,nights,channel,total_price
0,93,202,2025-01-29,12,OTA,"78,105.72"
1,465,203,2025-03-23,11,OTA,"63,440.30"
2,358,203,2025-02-27,9,OTA,"63,028.35"
3,3,203,2025-01-18,8,OTA,"55,366.08"
4,406,202,2025-02-28,8,Website,"49,434.00"
5,100,203,2025-02-01,7,Website,"49,022.05"
6,451,203,2025-04-28,10,OTA,"45,479.28"
7,281,202,2025-02-15,5,Phone,"38,105.38"
8,9,203,2025-01-11,6,OTA,"34,109.46"
9,147,202,2025-03-09,6,OTA,"32,230.97"


**Query 2** — SELECT / WHERE / ORDER BY : ผู้เข้าพักที่จองล่วงหน้านานที่สุดในช่วง High season

In [ ]:
q2 = q("""
    SELECT booking_id, nationality, room_type, check_in,
           lead_time_days, nights, ROUND(total_price, 2) AS total_price
    FROM {t}bookings{e}
    WHERE season = 'High' AND lead_time_days >= 60 AND status != 'CANCELLED'
    ORDER BY lead_time_days DESC
    LIMIT 15
""")
q2

,booking_id,nationality,room_type,check_in,lead_time_days,nights,total_price
0,1,Japanese,Standard,2025-01-31,146,1,"1,894.97"
1,8,German,Standard,2025-02-17,140,14,"38,064.18"
2,3,Australian,Suite,2025-01-18,124,8,"55,366.08"
3,10,Thai,Standard,2025-01-31,121,3,"8,156.61"
4,26,Russian,Standard,2025-02-23,121,4,"7,579.88"
5,4,German,Standard,2025-01-10,112,6,"13,841.52"
6,12,German,Standard,2025-01-25,110,6,"13,841.52"
7,44,Korean,Family,2025-02-28,110,1,"4,884.55"
8,23,Indian,Standard,2025-02-05,108,3,"5,684.91"
9,25,Japanese,Standard,2025-02-08,108,4,"7,579.88"


**Query 3** — SELECT / WHERE / ORDER BY : ใบจองที่ถูกยกเลิกและมูลค่าที่เสียโอกาส

In [ ]:
q3 = q("""
    SELECT booking_id, room_type, channel, check_in, nights, lead_time_days,
           ROUND(total_price, 2) AS lost_revenue
    FROM {t}bookings{e}
    WHERE status = 'CANCELLED'
    ORDER BY total_price DESC
    LIMIT 20
""")
q3

,booking_id,room_type,channel,check_in,nights,lead_time_days,lost_revenue
0,672,Suite,Website,2025-05-25,10,49,"49,170.35"
1,14,Suite,Website,2025-01-16,8,95,"48,774.88"
2,893,Suite,OTA,2025-06-21,8,22,"42,183.68"
3,126,Standard,OTA,2025-02-14,14,58,"38,064.18"
4,877,Standard,OTA,2025-06-15,14,19,"36,403.20"
5,77,Standard,OTA,2025-01-24,10,55,"32,956.00"
6,116,Standard,OTA,2025-03-27,14,102,"32,758.26"
7,848,Standard,Phone,2025-06-05,14,15,"32,296.88"
8,57,Deluxe,Website,2025-01-24,7,67,"29,495.62"
9,139,Family,OTA,2025-01-12,6,22,"29,130.75"


**Query 4** — GROUP BY + aggregate function (COUNT / SUM / AVG)

In [ ]:
q4 = q("""
    SELECT room_type,
           COUNT(*)                   AS total_bookings,
           SUM(nights)                AS total_nights,
           ROUND(AVG(nights), 2)      AS avg_nights,
           ROUND(SUM(total_price), 2) AS total_revenue,
           ROUND(AVG(total_price), 2) AS avg_per_booking,
           ROUND(SUM(net_revenue) / SUM(nights), 2) AS net_revenue_per_night
    FROM {t}bookings{e}
    WHERE status != 'CANCELLED'
    GROUP BY room_type
    ORDER BY total_revenue DESC
""")
q4

,room_type,total_bookings,total_nights,avg_nights,total_revenue,avg_per_booking,net_revenue_per_night
0,Deluxe,275,722,2.63,"2,233,591.55","8,122.15","2,830.08"
1,Standard,315,833,2.64,"1,746,236.45","5,543.61","1,908.47"
2,Suite,110,280,2.55,"1,589,850.43","14,453.19","5,112.86"
3,Family,104,252,2.42,"1,026,769.50","9,872.78","3,683.70"


**Query 5** — JOIN 3 ตารางพร้อมกัน : ลูกค้าสัญชาติใดจ่ายให้ห้องประเภทใดมากที่สุด

In [ ]:
q5 = q("""
    SELECT g.nationality,
           r.room_type,
           COUNT(b.booking_id)             AS bookings,
           ROUND(SUM(b.total_price), 2)    AS revenue,
           ROUND(AVG(b.nights), 2)         AS avg_nights,
           ROUND(AVG(b.lead_time_days), 1) AS avg_lead_time
    FROM {t}bookings{e} AS b
    JOIN {t}guests{e}   AS g ON b.guest_id = g.guest_id
    JOIN {t}rooms{e}    AS r ON b.room_id  = r.room_id
    WHERE b.status != 'CANCELLED'
    GROUP BY g.nationality, r.room_type
    HAVING bookings >= 5
    ORDER BY revenue DESC
    LIMIT 15
""")
q5

,nationality,room_type,bookings,revenue,avg_nights,avg_lead_time
0,German,Standard,23,"306,223.04",6.39,58.10
1,Chinese,Deluxe,44,"303,916.92",2.23,22.70
2,Russian,Deluxe,21,"276,750.46",4.14,49.80
3,German,Deluxe,19,"273,817.20",4.63,76.20
4,Chinese,Suite,21,"272,995.99",2.19,23.80
5,Thai,Deluxe,65,"268,392.97",1.37,11.20
6,British,Deluxe,14,"265,064.88",5.57,62.10
7,Chinese,Standard,52,"242,163.15",2.17,20.80
8,Russian,Standard,26,"220,549.79",4.00,57.80
9,Thai,Standard,67,"214,266.71",1.57,12.60


**Query 6** — JOIN + GROUP BY : ลูกค้า 5 อันดับแรกที่ใช้จ่ายมากที่สุด

In [ ]:
q6 = q("""
    SELECT g.name, g.nationality, g.member_tier,
           COUNT(b.booking_id)          AS times_stayed,
           SUM(b.nights)                AS total_nights,
           ROUND(SUM(b.total_price), 2) AS total_spend
    FROM {t}guests{e}   AS g
    JOIN {t}bookings{e} AS b ON g.guest_id = b.guest_id
    WHERE b.status != 'CANCELLED'
    GROUP BY g.guest_id, g.name, g.nationality, g.member_tier
    ORDER BY total_spend DESC
    LIMIT 5
""")
q6

,name,nationality,member_tier,times_stayed,total_nights,total_spend
0,Anna Fischer,German,Gold,7,46,"129,313.12"
1,Jessica Miller,American,Gold,5,31,"105,595.44"
2,Sarah Williams,American,Silver,3,22,"88,174.96"
3,James Davies,British,Regular,2,21,"83,625.85"
4,Rahul Gupta,Indian,Regular,1,12,"78,105.72"


**Query 7** — CASE WHEN + GROUP BY : อัตราการยกเลิกและค่าคอมมิชชันแยกตามช่องทาง

In [ ]:
q7 = q("""
    SELECT channel,
           COUNT(*) AS total_bookings,
           SUM(CASE WHEN status = 'CANCELLED' THEN 1 ELSE 0 END) AS cancelled,
           ROUND(100 * SUM(CASE WHEN status = 'CANCELLED' THEN 1 ELSE 0 END)
                 / COUNT(*), 1) AS cancel_rate_pct,
           ROUND(SUM(commission), 0) AS commission_paid
    FROM {t}bookings{e}
    GROUP BY channel
    ORDER BY cancel_rate_pct DESC
""")
q7

,channel,total_bookings,cancelled,cancel_rate_pct,commission_paid
0,OTA,553,118,21.00,"702,930.00"
1,Website,237,23,9.00,"38,353.00"
2,Phone,163,15,9.00,0.00
3,Walk-in,7,0,0.00,0.00


## ส่วนที่ 8 — วิเคราะห์ด้วย pandas

**คำถามทางธุรกิจที่กลุ่มตั้งไว้**
1. ห้องประเภทใดทำรายได้สุทธิมากที่สุด และคุ้มที่สุดเมื่อคิดต่อคืน
2. ช่องทางการจองใดทำเงินได้จริงมากที่สุดหลังหักค่าคอมมิชชัน
3. ฤดูกาลมีผลต่อราคาและปริมาณการจองอย่างไร
4. ลูกค้ากลุ่มใดใช้จ่ายมากที่สุด



In [ ]:
bookings = pd.read_csv("data/hotel_bookings.csv")
guests = pd.read_csv("data/hotel_guests.csv")
rooms = pd.read_csv("data/hotel_rooms.csv")

bookings["check_in"] = pd.to_datetime(bookings["check_in"])
bookings["booking_date"] = pd.to_datetime(bookings["booking_date"])

print("bookings:", bookings.shape, "| guests:", guests.shape, "| rooms:", rooms.shape)
bookings.head()

bookings: (960, 31) | guests: (748, 6) | rooms: (15, 7)


,booking_id,guest_id,room_id,room_number,room_type,booking_date,check_in,check_out,nights,lead_time_days,...,cancel_date,rate_per_night,room_charge,extra_charge,discount,service_charge,vat,total_price,commission,net_revenue
0,1,1,5,105,Standard,2024-09-07,2025-01-31,2025-02-01,1,146,...,NaN,"1,610.00","1,610.00",0,0.00,161.00,123.97,"1,894.97",284.25,"1,610.72"
1,2,2,10,110,Deluxe,2024-09-08,2025-03-07,2025-03-08,1,180,...,2025-02-12,"2,024.00","2,024.00",0,0.00,202.40,155.85,"2,382.25",357.34,"2,024.91"
2,3,2,13,203,Suite,2024-09-16,2025-01-18,2025-01-26,8,124,...,NaN,"4,830.00","38,640.00",8400,0.00,"4,704.00","3,622.08","55,366.08","8,304.91","47,061.17"
3,4,3,1,101,Standard,2024-09-20,2025-01-10,2025-01-16,6,112,...,NaN,"1,610.00","9,660.00",2100,0.00,"1,176.00",905.52,"13,841.52",276.83,"13,564.69"
4,5,4,6,106,Standard,2024-09-20,2025-01-02,2025-01-14,12,104,...,NaN,"1,610.00","19,320.00",4200,0.00,"2,352.00","1,811.04","27,683.04","4,152.46","23,530.58"


In [ ]:
bookings.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 960 entries, 0 to 959
Data columns (total 31 columns):
 #   Column                  Non-Null Count  Dtype         
---  ------                  --------------  -----         
 0   booking_id              960 non-null    int64         
 1   guest_id                960 non-null    int64         
 2   room_id                 960 non-null    int64         
 3   room_number             960 non-null    int64         
 4   room_type               960 non-null    object        
 5   booking_date            960 non-null    datetime64[ns]
 6   check_in                960 non-null    datetime64[ns]
 7   check_out               960 non-null    object        
 8   nights                  960 non-null    int64         
 9   lead_time_days          960 non-null    int64         
 10  adults                  960 non-null    int64         
 11  children                960 non-null    int64         
 12  booking_month           960 non-null    object    

In [ ]:
bookings[["nights", "lead_time_days", "rate_per_night",
          "total_price", "commission", "net_revenue"]].describe()

,nights,lead_time_days,rate_per_night,total_price,commission,net_revenue
count,960.00,960.00,960.00,960.00,960.00,960.00
mean,2.61,31.43,"2,350.05","8,215.77",772.17,"7,443.60"
std,2.54,33.62,"1,042.79","9,084.04","1,226.08","8,208.86"
min,1.00,0.00,"1,159.20","1,296.16",0.00,"1,159.72"
25%,1.00,8.00,"1,540.00","3,024.30",64.73,"2,749.03"
50%,2.00,21.00,"2,024.00","5,178.80",369.51,"4,661.51"
75%,3.00,41.25,"2,760.00","9,618.44",946.31,"8,753.32"
max,14.00,180.00,"6,300.00","78,105.72","11,715.86","66,389.86"


ตัดใบจองที่ยกเลิกออกก่อนวิเคราะห์รายได้ เนื่องจากไม่ได้สร้างรายได้จริง



In [ ]:
active = bookings[bookings["status"] != "CANCELLED"].copy()
cancel_rate = (bookings["status"] == "CANCELLED").mean() * 100

print(f"ใบจองทั้งหมด {len(bookings):,} ใบ | ใช้วิเคราะห์รายได้ {len(active):,} ใบ")
print(f"อัตราการยกเลิก {cancel_rate:.2f}%")
print(bookings["status"].value_counts())

ใบจองทั้งหมด 960 ใบ | ใช้วิเคราะห์รายได้ 804 ใบ
อัตราการยกเลิก 16.25%
status
CHECKED_OUT    777
CANCELLED      156
NO_SHOW         16
CONFIRMED       11
Name: count, dtype: int64


### คำถามที่ 1 : รายได้แยกตามประเภทห้อง (groupby + agg + sort_values)

In [ ]:
by_room_type = active.groupby("room_type").agg(
    total_bookings=("booking_id", "count"),
    total_nights=("nights", "sum"),
    avg_nights=("nights", "mean"),
    avg_rate=("rate_per_night", "mean"),
    total_revenue=("total_price", "sum"),
    net_revenue=("net_revenue", "sum"),
).reset_index()

by_room_type["net_per_night"] = by_room_type["net_revenue"] / by_room_type["total_nights"]
by_room_type = by_room_type.sort_values("net_revenue", ascending=False)
by_room_type

,room_type,total_bookings,total_nights,avg_nights,avg_rate,total_revenue,net_revenue,net_per_night
0,Deluxe,275,722,2.63,"2,311.80","2,233,591.55","2,043,321.05","2,830.08"
2,Standard,315,833,2.64,"1,475.93","1,746,236.45","1,589,757.85","1,908.47"
3,Suite,110,280,2.55,"4,354.56","1,589,850.43","1,431,600.85","5,112.86"
1,Family,104,252,2.42,"3,113.45","1,026,769.50","928,291.54","3,683.70"


### คำถามที่ 2 : ช่องทางการจอง

In [ ]:
by_channel = active.groupby("channel").agg(
    bookings=("booking_id", "count"),
    gross_revenue=("total_price", "sum"),
    commission_paid=("commission", "sum"),
    net_revenue=("net_revenue", "sum"),
    avg_lead_time=("lead_time_days", "mean"),
).reset_index()
by_channel["commission_pct"] = (by_channel["commission_paid"]
                                / by_channel["gross_revenue"] * 100)
by_channel.sort_values("net_revenue", ascending=False)

,channel,bookings,gross_revenue,commission_paid,net_revenue,avg_lead_time,commission_pct
0,OTA,435,"3,803,046.63","570,456.98","3,232,589.69",31.56,15.00
3,Website,214,"1,650,983.50","33,019.73","1,617,963.80",30.51,2.00
1,Phone,148,"1,099,263.10",0.00,"1,099,263.10",28.53,0.00
2,Walk-in,7,"43,154.70",0.00,"43,154.70",0.00,0.00


### คำถามที่ 3 : ฤดูกาลและรายเดือน

In [ ]:
by_season = active.groupby("season").agg(
    bookings=("booking_id", "count"),
    total_nights=("nights", "sum"),
    avg_rate=("rate_per_night", "mean"),
    revenue=("total_price", "sum"),
).reset_index().sort_values("avg_rate", ascending=False)

by_month = active.groupby("booking_month").agg(
    bookings=("booking_id", "count"),
    room_nights=("nights", "sum"),
    avg_rate=("rate_per_night", "mean"),
    revenue=("total_price", "sum"),
).reset_index().sort_values("booking_month")
by_month["occupancy_%"] = by_month["room_nights"] / (len(rooms) * 30) * 100

display(by_season)
by_month

,season,bookings,total_nights,avg_rate,revenue
0,High,276,753,"2,845.63","2,809,343.92"
2,Normal,270,701,"2,228.65","2,094,781.58"
1,Low,258,633,"2,001.29","1,692,322.43"


,booking_month,bookings,room_nights,avg_rate,revenue,occupancy_%
0,2025-01,140,411,"2,893.39","1,549,074.73",91.33
1,2025-02,136,342,"2,796.47","1,260,269.19",76.00
2,2025-03,134,344,"2,176.36","1,014,034.96",76.44
3,2025-04,136,357,"2,280.18","1,080,746.62",79.33
4,2025-05,137,341,"2,007.93","896,624.44",75.78
5,2025-06,121,292,"1,993.78","795,697.99",64.89


### คำถามที่ 4 : ลูกค้าและตลาดที่ใช้จ่ายมากที่สุด (sort_values)

In [ ]:
spend = active.groupby("guest_id").agg(
    total_spend=("total_price", "sum"),
    times_stayed=("booking_id", "count"),
    total_nights=("nights", "sum"),
).reset_index()

top_guests = (spend.merge(guests[["guest_id", "name", "nationality", "member_tier"]],
                          on="guest_id")
              .sort_values("total_spend", ascending=False).head(5))
display(top_guests)

by_nat = active.groupby("nationality").agg(
    bookings=("booking_id", "count"),
    revenue=("total_price", "sum"),
    avg_spend=("total_price", "mean"),
    avg_nights=("nights", "mean"),
    avg_lead_time=("lead_time_days", "mean"),
).reset_index().sort_values("revenue", ascending=False)
by_nat

,guest_id,total_spend,times_stayed,total_nights,name,nationality,member_tier
2,3,"129,313.12",7,46,Anna Fischer,German,Gold
4,5,"105,595.44",5,31,Jessica Miller,American,Gold
34,37,"88,174.96",3,22,Sarah Williams,American,Silver
68,75,"83,625.85",2,21,James Davies,British,Regular
49,53,"78,105.72",1,12,Rahul Gupta,Indian,Regular


,nationality,bookings,revenue,avg_spend,avg_nights,avg_lead_time
3,Chinese,127,"900,359.68","7,089.45",2.17,22.74
11,Thai,191,"882,907.27","4,622.55",1.47,13.59
4,German,54,"739,522.57","13,694.86",5.04,65.98
9,Russian,59,"685,364.25","11,616.34",3.92,57.63
2,British,41,"662,511.65","16,158.82",4.59,60.46
8,Malaysian,90,"542,044.33","6,022.71",1.79,17.96
0,American,35,"461,520.02","13,186.29",4.29,53.51
5,Indian,51,"442,780.01","8,681.96",2.51,27.94
7,Korean,61,"384,915.25","6,310.09",2.25,25.46
1,Australian,29,"384,216.11","13,248.83",3.66,38.31
